In [71]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

In [72]:
import os
import requests
import pandas as pd
import time
from dotenv import load_dotenv
from datetime import datetime, timedelta
load_dotenv()  # 환경변수 불러오기

True

# API 정보 불러오기

In [114]:
# 하이퍼 파라미터
START_YEAR = 2024
END_YEAR = 2024
ITEM_CODE = '211'  # 111 : '쌀', 211 : '배추', 245 : '양파', 214 : '상추', 411 : '사과'
MARKET_TYPE ='02'  # 상품분류: 01=소매, 02=도매

# KAMIS 인증 정보
CERT_KEY = os.getenv('kamis_key')
CERT_ID = os.getenv('kamis_id')

#아이템 코드
item_map = {'111' : '쌀',
           '211' : '배추',
           '245' : '양파',
           '214' : '상추',
           '411' : '사과'} 

# 실행

In [103]:
p_item_code in ('211', '245', '214')

True

In [117]:
print(p_startday, p_endday, p_item_code, p_item_category_code, item_map[p_item_code])

2024-01-01 2024-01-31 211 200 배추


In [121]:
url

'https://www.kamis.or.kr/service/price/xml.do'

In [122]:
# 대분류 지정
p_item_code = ITEM_CODE
if p_item_code == '111':
    p_item_category_code = '100'
elif p_item_code in ('211', '245', '214'):
    p_item_category_code = '200'
elif p_item_code == '411':
    p_item_category_code = '400'
else:
    print('대분류에 없는 코드입니다')
    
url = 'https://www.kamis.or.kr/service/price/xml.do'
data_list = []
year = START_YEAR
for i in range(END_YEAR-START_YEAR+1):    
    p_startday = f'{year}-01-01'
    p_endday = f'{year}-01-31'
    if year == datetime.today().strftime('%Y'): # 올해라면, 전일자 까지만 뽑기
        p_endday= (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')

    params = {
        'action': 'periodWholesaleProductList',  # 신) 일별 도매 가격 자료
        'p_cert_key': CERT_KEY,
        'p_cert_id': CERT_ID,
        'p_returntype': 'json',
        'p_product_cls_code': MARKET_TYPE,       # 상품분류: 01=소매, 02=도매
        'p_item_category_code': p_item_category_code,    # 대분류 코드 예: 200=과일류
        'p_item_code': p_item_code,             # 품목코드 예: 211=사과
        # 'p_kind_code': '05',              # 품종코드 예: 05=홍로
        # 'p_county_code': '1101',          # 지역코드 예: 서울=1101
        #'p_convert_kg_yn': 'N',
        'p_startday': p_startday,
        'p_endday': p_endday
    }

    # ✅ API 호출
    response = requests.get(url, params=params)

    # ✅ 결과 확인 및 데이터 추가
    if response.status_code == 200:
        json_data = response.json()
        data = json_data.get('data')
        data_list.extend(data.get('item'))
    else:
        print(f"API 호출 실패 - 코드 : {response.status_code}, 작업중 : {item_map[p_item_code]} - {p_startday}~{p_endday}" )
        break

    # 다음 사이클로 이동
    year += 1
    print(f'작업완료 - {item_map[p_item_code]}, 기간 : {p_startday}~{p_endday}')
    time.sleep(1)
df = pd.DataFrame(data_list)
# values = {'itemname' : item_map[p_item_code], 'kindname' : '-', 'marketname' : '-'}
# df.fillna(value= values, inplace=True)
df.to_csv(f'data/kamis_api_도매_{item_map[p_item_code]}.csv', encoding='cp949')
print('=========저장완료==========')

작업완료 - 배추, 기간 : 2024-01-01~2024-01-31
=========저장완료==========


In [96]:
print("👉 호출 URL:", response.url)
print("👉 응답 상태 코드:", response.status_code)
print("👉 응답 내용 요약:", response.text[:500])

👉 호출 URL: https://www.kamis.or.kr/service/price/xml.do?action=periodWholesaleProductList&p_cert_key=fcd4157a-7a04-411f-93c3-04af27715ed5&p_cert_id=5858&p_returntype=json&p_product_cls_code=01&p_item_category_code=200&p_item_code=211&p_convert_kg_yn=Y&p_startday=2024-01-01&p_endday=2024-12-31
👉 응답 상태 코드: 500
👉 응답 내용 요약: {"timestamp":"2025-06-20 18:14:49.241","requestId":"54d610fb-43063545","path":"/service/price/xml.do","status":500,"error":"Internal Server Error","errorCode":"ione.apigtw.error.apisvc","message":{"timestamp":"2025-06-20 18:14:49.238","system":"DEFAULT","guid":"202506201814192319a5103d06b7f4513a05ef6a4dabc633a","path":"/service/price/xml.do","status":500,"error":"Internal Server Error","code":"SYSE500000","message":"시스템 오류가 발생을 하였습니다.[Cause : Could not open JDBC Connection for transaction; neste


In [120]:
response.text

'{"condition":{"item":{"p_startday":"2024-01-01","p_endday":"2024-01-31","p_itemcategorycode":"100","p_itemcode":"111","p_kindcode":"01","p_productrankcode":"04","p_countycode":null,"p_convert_kg_yn":"N","p_key":"fcd4157a-7a04-411f-93c3-04af27715ed5","p_id":"5858","p_returntype":"json"}},"data":{"error_code":"000","item":[{"itemname":null,"kindname":null,"countyname":"평균","marketname":null,"yyyy":"2024","regday":"01/02","price":"49,920"},{"itemname":null,"kindname":null,"countyname":"평균","marketname":null,"yyyy":"2024","regday":"01/03","price":"49,920"},{"itemname":null,"kindname":null,"countyname":"평균","marketname":null,"yyyy":"2024","regday":"01/04","price":"49,920"},{"itemname":null,"kindname":null,"countyname":"평균","marketname":null,"yyyy":"2024","regday":"01/05","price":"49,920"},{"itemname":null,"kindname":null,"countyname":"평균","marketname":null,"yyyy":"2024","regday":"01/08","price":"49,840"},{"itemname":null,"kindname":null,"countyname":"평균","marketname":null,"yyyy":"2024","re

In [84]:
cols_to_fill = ['itemname', 'kindname']
df['itemname'] = df['itemname'].fillna(method='bfill')
df['kindname'] = df['kindname'].fillna(method='bfill')
df['marketname'] = df['marketname'].fillna()

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1708 entries, 0 to 1707
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   itemname    1708 non-null   object
 1   kindname    1708 non-null   object
 2   countyname  1708 non-null   object
 3   marketname  1220 non-null   object
 4   yyyy        1708 non-null   object
 5   regday      1708 non-null   object
 6   price       1708 non-null   object
dtypes: object(7)
memory usage: 93.5+ KB


In [ ]:
df.to_csv(f'data/kamis_api_도매_{item_map[p_item_code]}.csv', encoding='cp949')

In [88]:
data_list = []

In [105]:
data.get('item')[-1]

{'itemname': '쌀',
 'kindname': '20kg(1kg)',
 'countyname': '대전',
 'marketname': '인동',
 'yyyy': '2024',
 'regday': '01/31',
 'price': '2,490'}